In [0]:
USE CATALOG IDENTIFIER(:catalog_name);
USE SCHEMA IDENTIFIER(:schema_name);

-- ============================================================
-- Epic on FHIR — Full Endpoint Test Flow (via ai_query)
-- Mirrors: epic-on-fhir-example notebook
-- Endpoint: sandbox_epic_on_fhir_requests
-- ============================================================

-- Session variables for chaining responses between steps
DECLARE OR REPLACE VARIABLE patient_id STRING;
DECLARE OR REPLACE VARIABLE encounter_id STRING;
DECLARE OR REPLACE VARIABLE observation_id STRING;
DECLARE OR REPLACE VARIABLE allergy_id STRING;

-- ─── Step 1: Search Patient by External Identifier ───────────
-- GET Patient?identifier=EXTERNAL|Z6129
-- Extract the FHIR STU3 ID from the Bundle's first entry
-- (notebook cell 16 overrides the DSTU2 'FHIR' ID with the STU3 ID)
SET VARIABLE patient_id = (
  WITH raw AS (
    SELECT ai_query(
      'sandbox_epic_on_fhir_requests',
      request => named_struct(
        'http_method', 'get',
        'resource', 'Patient',
        'action', '?identifier=EXTERNAL|Z6129',
        'data', CAST(NULL AS STRING)
      ),
      returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
    ) AS resp
  )
  SELECT
    filter(
      from_json(
        CAST(parse_json(resp.response_text):entry[0].resource.identifier AS STRING),
        'ARRAY<STRUCT<type:STRUCT<text:STRING>, value:STRING>>'
      ),
      x -> x.type.text = 'FHIR STU3'
    )[0].value
  FROM raw
);

-- Verify: show extracted patient_id
SELECT 'Step 1: Patient Search' AS step, patient_id;

-- ─── Step 2: Patient Clinical Summary ────────────────────────
-- GET Patient/{patient_id}/$summary
WITH raw AS (
  SELECT ai_query(
    'sandbox_epic_on_fhir_requests',
    request => named_struct(
      'http_method', 'get',
      'resource', 'Patient',
      'action', patient_id || '/$summary',
      'data', CAST(NULL AS STRING)
    ),
    returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
  ) AS resp
)
SELECT
  'Step 2: Patient/$summary' AS step,
  patient_id,
  resp.response_status_code,
  resp.response_url,
  resp.response_time_seconds,
  parse_json(resp.response_text) AS response_text
FROM raw;

-- ─── Step 3: Search Encounters for Patient ───────────────────
-- GET Encounter?patient=Patient/{patient_id}&date=le{today}
-- Extract encounter_id from the first result
SET VARIABLE encounter_id = (
  WITH raw AS (
    SELECT ai_query(
      'sandbox_epic_on_fhir_requests',
      request => named_struct(
        'http_method', 'get',
        'resource', 'Encounter',
        'action', '?patient=Patient/' || patient_id || '&date=le' || CAST(current_date() AS STRING),
        'data', CAST(NULL AS STRING)
      ),
      returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
    ) AS resp
  )
  SELECT parse_json(resp.response_text):entry[0].resource.id::STRING
  FROM raw
);

-- Verify: show extracted encounter_id
SELECT 'Step 3: Encounter Search' AS step, patient_id, encounter_id;

-- ─── Step 4: Create Observation — Heart Rate Vital Sign ──────
-- POST Observation (Epic Flowsheet code 8 = Heart Rate)
-- Uses current_timestamp for effectiveDateTime (unique per run)
SET VARIABLE observation_id = (
  WITH raw AS (
    SELECT ai_query(
      'sandbox_epic_on_fhir_requests',
      request => named_struct(
        'http_method', 'post',
        'resource', 'Observation',
        'action', '',
        'data', format_string(
          concat(
            '{"resourceType":"Observation","status":"final",',
            '"category":[{"coding":[{"system":"http://hl7.org/fhir/observation-category","code":"vital-signs","display":"Vital Signs"}],"text":"Vital Signs"}],',
            '"code":{"coding":[{"system":"urn:oid:1.2.840.114350.1.13.0.1.7.2.707679","code":"8","display":"Heart Rate"}],"text":"Heart Rate"},',
            '"subject":{"reference":"Patient/%s"},',
            '"encounter":{"reference":"Encounter/%s"},',
            '"effectiveDateTime":"%s",',
            '"valueQuantity":{"value":75}}'
          ),
          patient_id,
          encounter_id,
          date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss'Z'")
        )
      ),
      returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
    ) AS resp
  )
  SELECT
    CASE
      WHEN resp.response_status_code = 201
      THEN element_at(split(parse_json(resp.response_headers):Location::STRING, '/'), -1)
    END
  FROM raw
);

-- ─── Step 5: Verify Observation (Read-Back) ──────────────────
-- GET Observation/{observation_id}
WITH raw AS (
  SELECT ai_query(
    'sandbox_epic_on_fhir_requests',
    request => named_struct(
      'http_method', 'get',
      'resource', 'Observation',
      'action', COALESCE(observation_id, ''),
      'data', CAST(NULL AS STRING)
    ),
    returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
  ) AS resp
)
SELECT
  'Step 5: GET Observation (verify)' AS step,
  observation_id,
  resp.response_status_code,
  resp.response_url,
  resp.response_time_seconds,
  parse_json(resp.response_text) AS response_text
FROM raw;

-- ─── Step 6: Create AllergyIntolerance — Penicillin ──────────
-- POST AllergyIntolerance (RxNorm 7980 = Penicillin G)
-- Practitioner eM5CWtq15N0WJeuCet5bJlQ3 = Physician Family Medicine
SET VARIABLE allergy_id = (
  WITH raw AS (
    SELECT ai_query(
      'sandbox_epic_on_fhir_requests',
      request => named_struct(
        'http_method', 'post',
        'resource', 'AllergyIntolerance',
        'action', '',
        'data', format_string(
          concat(
            '{"resourceType":"AllergyIntolerance",',
            '"clinicalStatus":{"coding":[{"system":"http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical","code":"active","display":"Active"}],"text":"Active"},',
            '"verificationStatus":{"coding":[{"system":"http://terminology.hl7.org/CodeSystem/allergyintolerance-verification","code":"unconfirmed","display":"Unconfirmed"}],"text":"Unconfirmed"},',
            '"type":"allergy","category":["medication"],"criticality":"low",',
            '"code":{"coding":[{"system":"http://www.nlm.nih.gov/research/umls/rxnorm","code":"7980","display":"Penicillin G"}],"text":"Penicillin"},',
            '"patient":{"reference":"Patient/%s"},',
            '"recorder":{"reference":"Practitioner/eM5CWtq15N0WJeuCet5bJlQ3","display":"Physician Family Medicine, MD"},',
            '"recordedDate":"2024-01-15",',
            '"reaction":[{"manifestation":[{"coding":[{"system":"http://snomed.info/sct","code":"247472004","display":"Hives"}],"text":"Hives"}]}]}'
          ),
          patient_id
        )
      ),
      returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
    ) AS resp
  )
  SELECT
    CASE
      WHEN resp.response_status_code = 201
      THEN element_at(split(parse_json(resp.response_headers):Location::STRING, '/'), -1)
    END
  FROM raw
);

-- ─── Step 7: Verify AllergyIntolerance (Read-Back) ───────────
-- GET AllergyIntolerance/{allergy_id}
WITH raw AS (
  SELECT ai_query(
    'sandbox_epic_on_fhir_requests',
    request => named_struct(
      'http_method', 'get',
      'resource', 'AllergyIntolerance',
      'action', COALESCE(allergy_id, ''),
      'data', CAST(NULL AS STRING)
    ),
    returnType => 'STRUCT<response_headers:STRING, response_url:STRING, response_time_seconds:DOUBLE, response_status_code:INT, response_text:STRING>'
  ) AS resp
)
SELECT
  'Step 7: GET AllergyIntolerance (verify)' AS step,
  allergy_id,
  resp.response_status_code,
  resp.response_url,
  resp.response_time_seconds,
  parse_json(resp.response_text) AS response_text
FROM raw;

-- ─── Summary ─────────────────────────────────────────────────
SELECT
  patient_id,
  encounter_id,
  observation_id,
  allergy_id,
  CASE
    WHEN patient_id IS NOT NULL AND encounter_id IS NOT NULL
     AND observation_id IS NOT NULL AND allergy_id IS NOT NULL
    THEN 'ALL STEPS PASSED'
    ELSE concat_ws(', ',
      CASE WHEN patient_id IS NULL THEN 'Patient Search FAILED' END,
      CASE WHEN encounter_id IS NULL THEN 'Encounter Search FAILED' END,
      CASE WHEN observation_id IS NULL THEN 'Observation POST FAILED' END,
      CASE WHEN allergy_id IS NULL THEN 'AllergyIntolerance POST FAILED' END
    )
  END AS result;